In [1]:
import pandas as pd

In [2]:
!pip install openpyxl


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python3.12 -m pip install --upgrade pip


In [3]:
df= pd.read_excel('Medical_Data_For_Fine_Tuning.xlsx')

In [4]:
df.head()

,Case_ID,Input_Narrative,Structured_Output_JSON
0,PV-REAL-001,A 54-year-old male treated with Atorvastatin f...,"{""seriousness"": true, ""meddra_terms"": [""Myalgi..."
1,PV-REAL-002,A 61-year-old female with type 2 diabetes on M...,"{""seriousness"": true, ""meddra_terms"": [""Somnol..."
2,PV-REAL-003,A 45-year-old male experienced acute tongue sw...,"{""seriousness"": true, ""meddra_terms"": [""Angioe..."
3,PV-REAL-004,A 28-year-old female patient developed anaphyl...,"{""seriousness"": true, ""meddra_terms"": [""Anaphy..."
4,PV-REAL-005,A 68-year-old male on chronic Warfarin therapy...,"{""seriousness"": true, ""meddra_terms"": [""Hematu..."


In [5]:
!pip install -r requirements.txt


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python3.12 -m pip install --upgrade pip


In [ ]:
import os
os.environ["HSA_OVERRIDE_GFX_VERSION"] = "11.0.0" # AMD ROCm hardware spoof
os.environ["HF_HUB_DISABLE_XET"] = "1"

import torch
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from peft import LoraConfig, get_peft_model
from datasets import Dataset
from trl import SFTTrainer

model_id = "Qwen/Qwen2.5-3B-Instruct"

# 1. Initialize 16-bit precision model & tokenizer
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=dtype, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

# 2. Configure Standard PEFT LoRA Adapters
peft_config = LoraConfig(
    r=16, lora_alpha=16, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)
model = get_peft_model(model, peft_config)
model.gradient_checkpointing_enable() # Drastically cuts VRAM usage

# 3. Import and Parse your Excel Sheets
df = pd.read_excel("Medical_Data_For_Fine_Tuning.xlsx")
dataset = Dataset.from_pandas(df)

# 4. Map Rows directly into Qwen Chat Templates
def format_prompts(examples):
    texts = []
    for inp, out in zip(examples["Input_Narrative"], examples["Structured_Output_JSON"]):
        messages = [
            {"role": "system", "content": "Extract PV data into a single-line flattened JSON string."},
            {"role": "user", "content": str(inp)},
            {"role": "assistant", "content": str(out)}
        ]
        texts.append(tokenizer.apply_chat_template(messages, tokenize=False))
    return {"text": texts}

dataset = dataset.map(format_prompts, batched=True)

# 5. Define Hugging Face Native Training Hyperparameters
training_args = TrainingArguments(
    output_dir="hf_outputs",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    warmup_steps=5,
    max_steps=60,
    learning_rate=2e-4,
    fp16=dtype == torch.float16,
    bf16=dtype == torch.bfloat16,
    logging_steps=1,
    report_to="none"
)

# 6. Initialize SFT Engine and Execute Train loop
trainer = SFTTrainer(
    model=model, train_dataset=dataset, dataset_text_field="text",
    max_seq_length=2048, args=training_args
)
trainer.train()

`torch_dtype` is deprecated! Use `dtype` instead!
